## Persistent Landing - Delta Lake (Non-Structured)

Even though Delta tables are typically used to store structured data, they can also be used to store metadata extracted from unstructured or semi-structured data.

**Importing Useful Libraries**

In [1]:
import os
import boto3
import duckdb
import hashlib
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from deltalake import DeltaTable, write_deltalake
import polars as pl
from PIL import Image
from PIL.ExifTags import TAGS
import pandas as pd
import json
import io

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
MINIO_ROLE = "writer"
if MINIO_ROLE == "admin":
    access_key = os.getenv("MINIO_ACCESS_KEY")
    secret_key = os.getenv("MINIO_SECRET_KEY")
else:
    role_prefix = MINIO_ROLE.upper()
    access_key = os.getenv(f"MINIO_{role_prefix}_ACCESS_KEY")
    secret_key = os.getenv(f"MINIO_{role_prefix}_SECRET_KEY")
if not endpoint or not access_key or not secret_key:
    raise RuntimeError(f"Missing MinIO {MINIO_ROLE} credentials in environment")


In [2]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)

In [3]:
def get_deep_keys(data,level=0):
    # If it's a list, dive into the first element
    if isinstance(data, list) and len(data) > 0:
        return get_deep_keys(data[0],level+1)

    # If it's finally a dictionary, return the keys
    if isinstance(data, dict):
        return list(data.keys()),level

    # If it's a primitive (like a string or number) or empty
    return [],level

def extract_timestamp_from_filename(filename):
    # Strip extension and split by underscore
    name_part = os.path.splitext(filename)[0]
    raw_ts = name_part.split('_')[-1]

    try:
        # Convert string epoch to a readable datetime object
        dt_object = datetime.fromtimestamp(int(raw_ts))
        return dt_object
    except (ValueError, IndexError):
        # Fallback if the filename doesn't follow the pattern
        return datetime.now()
        
storage_options = {
    "AWS_ACCESS_KEY_ID": access_key,
    "AWS_SECRET_ACCESS_KEY": secret_key,
    "AWS_ENDPOINT_URL": endpoint,
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    "AWS_S3_ADDRESSING_STYLE": "path",
    "AWS_ALLOW_HTTP": "true",
    "region": "us-east-1"
}
CATALOG_STRING_COLUMNS = [
    "file_id",
    "file_path",
    "source_type",
    "file_type",
    "event_time",
    "metadata_blob",
]


def normalize_catalog_df(df: pd.DataFrame) -> pd.DataFrame:
    """Keep Delta catalog writes stable even when a batch has all-null values."""
    normalized = df.copy()
    for column in CATALOG_STRING_COLUMNS:
        if column not in normalized.columns:
            normalized[column] = ""
        normalized[column] = normalized[column].fillna("").astype(str)

    if "record_count" not in normalized.columns:
        normalized["record_count"] = 0
    normalized["record_count"] = normalized["record_count"].fillna(0).astype("int64")

    if "processed_at" not in normalized.columns:
        normalized["processed_at"] = pd.Timestamp.now()
    normalized["processed_at"] = pd.to_datetime(normalized["processed_at"], errors="coerce").fillna(pd.Timestamp.now())

    return normalized[[
        "file_id",
        "source_type",
        "file_path",
        "file_type",
        "event_time",
        "record_count",
        "metadata_blob",
        "processed_at",
    ]]


**Semistructured Data**

In [4]:
def process_json(bucket, prefix):
    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            src_key = obj["Key"]

            # 1. Skip directories and empty files
            if src_key.endswith("/") or obj['Size'] == 0:
                continue

            print(f"Processing: {src_key}")

            # 2. Read and Parse
            response = s3.get_object(Bucket=bucket, Key=src_key)
            content = response['Body'].read().decode('utf-8')
            data = json.loads(content)

            # 3. Extract Deep Metadata
            keys_list, level = get_deep_keys(data)

            # 4. Correct Record Counting (Flattening for the count)
            # This ensures [[{}, {}]] returns 2, not 1
            temp_data = data
            for _ in range(level):
                if isinstance(temp_data, list) and len(temp_data) > 0:
                    temp_data = [item for sublist in temp_data for item in (sublist if isinstance(sublist, list) else [sublist])]
            record_count = len(temp_data)

            # 5. Build the Metadata Blob (The "Table inside a Table")
            # This blob changes structure based on file type
            metadata_blob = {
                "nesting_level": level,
                "schema_keys": keys_list,
                "file_size_bytes": obj['Size']
            }

            # 6. Prepare Final Catalog Row
            filename = os.path.basename(src_key)
            metadata_row = pd.DataFrame([{
                "file_id": filename,
                "source_type": src_key.split('/')[2],
                "file_path": src_key,
                "file_type": "JSON",
                "event_time": extract_timestamp_from_filename(filename),
                "record_count": record_count,
                "metadata_blob": json.dumps(metadata_blob),  # The flexible packet
                "processed_at": pd.Timestamp.now()
            }])

            # 7. Append to Master Catalog
            metadata_row = normalize_catalog_df(metadata_row)
            write_deltalake(
                "s3://landing-zone/persistent-landing/structured/file_catalog/",
                metadata_row,
                mode="append",
                schema_mode="merge",
                storage_options=storage_options
            )

In [5]:
process_json("landing-zone", "persistent-landing/semistructured/")


Processing: persistent-landing/semistructured/airquality-barcelona.json
Processing: persistent-landing/semistructured/airquality-barcelona/2026_05_24/1779621760.json
Processing: persistent-landing/semistructured/airquality-barcelona/2026_05_24/1779622038.json
Processing: persistent-landing/semistructured/weather-barcelona.json
Processing: persistent-landing/semistructured/weather-barcelona/2026_05_24/1779621760.json
Processing: persistent-landing/semistructured/weather-barcelona/2026_05_24/1779622038.json


In [6]:
# Connect to DuckDB and configure S3 secret for MinIO
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET secret (
    TYPE s3,
    PROVIDER config,
    ENDPOINT '{endpoint.replace("http://", "").replace("https://", "")}',
    KEY_ID '{access_key}',
    SECRET '{secret_key}',
    URL_STYLE 'path',
    USE_SSL false
);
""")

In [7]:
# 1. Show all columns (don't hide the middle ones)
pd.set_option('display.max_columns', None)

# 2. Show the full content of each cell (don't truncate long JSON/strings)
pd.set_option('display.max_colwidth', None)

# 3. Show all rows (optional: only use if the table is small, e.g., < 100 rows)
# pd.set_option('display.max_rows', None)

# Execute and display
query = "SELECT * FROM delta_scan('s3://landing-zone/persistent-landing/structured/file_catalog/')"
df_view = con.execute(query).df()
display(df_view)

,file_id,file_path,source_type,file_type,event_time,record_count,metadata_blob,processed_at
0,1779622038.json,persistent-landing/semistructured/weather-barcelona/2026_05_24/1779622038.json,weather-barcelona,JSON,2026-05-24 11:27:18.000000,5,"{""nesting_level"": 1, ""schema_keys"": [""time"", ""interval"", ""temperature"", ""windspeed"", ""winddirection"", ""is_day"", ""weathercode""], ""file_size_bytes"": 695}",2026-05-24 11:32:48.959459
1,1779621760.json,persistent-landing/semistructured/weather-barcelona/2026_05_24/1779621760.json,weather-barcelona,JSON,2026-05-24 11:22:40.000000,6,"{""nesting_level"": 1, ""schema_keys"": [""time"", ""interval"", ""temperature"", ""windspeed"", ""winddirection"", ""is_day"", ""weathercode""], ""file_size_bytes"": 834}",2026-05-24 11:32:48.755846
2,weather-barcelona.json,persistent-landing/semistructured/weather-barcelona.json,weather-barcelona.json,JSON,2026-05-24 11:32:48.557731,5,"{""nesting_level"": 1, ""schema_keys"": [""time"", ""interval"", ""temperature"", ""windspeed"", ""winddirection"", ""is_day"", ""weathercode""], ""file_size_bytes"": 695}",2026-05-24 11:32:48.557757
3,1779622038.json,persistent-landing/semistructured/airquality-barcelona/2026_05_24/1779622038.json,airquality-barcelona,JSON,2026-05-24 11:27:18.000000,15,"{""nesting_level"": 2, ""schema_keys"": [""id"", ""name"", ""locality"", ""timezone"", ""country"", ""owner"", ""provider"", ""isMobile"", ""isMonitor"", ""instruments"", ""sensors"", ""coordinates"", ""licenses"", ""bounds"", ""distance"", ""datetimeFirst"", ""datetimeLast""], ""file_size_bytes"": 27535}",2026-05-24 11:32:48.365120
4,1779621760.json,persistent-landing/semistructured/airquality-barcelona/2026_05_24/1779621760.json,airquality-barcelona,JSON,2026-05-24 11:22:40.000000,18,"{""nesting_level"": 2, ""schema_keys"": [""id"", ""name"", ""locality"", ""timezone"", ""country"", ""owner"", ""provider"", ""isMobile"", ""isMonitor"", ""instruments"", ""sensors"", ""coordinates"", ""licenses"", ""bounds"", ""distance"", ""datetimeFirst"", ""datetimeLast""], ""file_size_bytes"": 33042}",2026-05-24 11:32:48.185131
5,airquality-barcelona.json,persistent-landing/semistructured/airquality-barcelona.json,airquality-barcelona.json,JSON,2026-05-24 11:32:47.994066,15,"{""nesting_level"": 2, ""schema_keys"": [""id"", ""name"", ""locality"", ""timezone"", ""country"", ""owner"", ""provider"", ""isMobile"", ""isMonitor"", ""instruments"", ""sensors"", ""coordinates"", ""licenses"", ""bounds"", ""distance"", ""datetimeFirst"", ""datetimeLast""], ""file_size_bytes"": 27535}",2026-05-24 11:32:47.994086
6,1779621760.json,persistent-landing/semistructured/airquality-barcelona/2026_05_24/1779621760.json,airquality-barcelona,JSON,2026-05-24 11:22:40.000000,18,"{""nesting_level"": 2, ""schema_keys"": [""id"", ""name"", ""locality"", ""timezone"", ""country"", ""owner"", ""provider"", ""isMobile"", ""isMonitor"", ""instruments"", ""sensors"", ""coordinates"", ""licenses"", ""bounds"", ""distance"", ""datetimeFirst"", ""datetimeLast""], ""file_size_bytes"": 33042}",2026-05-24 11:27:41.701076
7,1779622038.json,persistent-landing/semistructured/airquality-barcelona/2026_05_24/1779622038.json,airquality-barcelona,JSON,2026-05-24 11:27:18.000000,15,"{""nesting_level"": 2, ""schema_keys"": [""id"", ""name"", ""locality"", ""timezone"", ""country"", ""owner"", ""provider"", ""isMobile"", ""isMonitor"", ""instruments"", ""sensors"", ""coordinates"", ""licenses"", ""bounds"", ""distance"", ""datetimeFirst"", ""datetimeLast""], ""file_size_bytes"": 27535}",2026-05-24 11:27:41.706471
8,1779621760.json,persistent-landing/semistructured/weather-barcelona/2026_05_24/1779621760.json,weather-barcelona,JSON,2026-05-24 11:22:40.000000,6,"{""nesting_level"": 1, ""schema_keys"": [""time"", ""interval"", ""temperature"", ""windspeed"", ""winddirection"", ""is_day"", ""weathercode""], ""file_size_bytes"": 834}",2026-05-24 11:27:41.424775
9,1779622038.json,persistent-landing/semistructured/weather-barce

**Unstructured Data**

In [8]:
def process_image(bucket, prefix):
    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        page_records = []

        for obj in page.get("Contents", []):
            try:
                src_key = obj["Key"]
                # 1. Skip directories and empty files
                if src_key.endswith("/") or obj['Size'] == 0:
                    continue

                # 2. Read and Parse
                response = s3.get_object(Bucket=bucket, Key=src_key)
                content = response['Body'].read()

                img = Image.open(io.BytesIO(content))
                metadata = response.get('Metadata', {})
                width, height = img.size

                metadata_blob = {
                                    "label": metadata.get('label'),
                                    "url": metadata.get('url'),
                                    "file_size_bytes": obj['Size'],
                                    "content_type": response.get('ContentType'),
                                    "width": width,
                                    "height": height,
                                    "aspect_ratio": round(width / height, 2) if height > 0 else 0,
                                    "image_mode": img.mode,
                                    "is_corrupted": False,
                                    "md5": hashlib.md5(content).hexdigest() # for duplicate detection
                                }

            except Exception as e:
                print(f"Error parsing image {src_key}: {e}")
                metadata_blob.update({
                    "is_corrupted": True,
                    "error_msg": str(e),
                    "width": 0, "height": 0, "aspect_ratio": 0, "image_mode": "unknown"
                })



            # 6. Prepare Final Catalog Row
            filename = os.path.basename(src_key)
            metadata_row = {
                "file_id": filename,
                "file_path": src_key,
                "source_type": metadata.get('source') or src_key.split('/')[2],
                "file_type": "Image",
                "event_time": extract_timestamp_from_filename(filename),
                "record_count": 1, # For image data it alaways 1
                "metadata_blob": json.dumps(metadata_blob),  # The flexible packet
                "processed_at": pd.Timestamp.now()
            }

            page_records.append(metadata_row)

        # 7. Append to Master Catalog
        if page_records:
            metadata_df = normalize_catalog_df(pd.DataFrame(page_records))
            write_deltalake(
                "s3://landing-zone/persistent-landing/structured/file_catalog/",
                metadata_df,
                mode="append",
                schema_mode="merge",
                storage_options=storage_options
            )
        print(f"Batch uploaded: {len(page_records)} images.")

In [9]:
process_image("landing-zone","persistent-landing/unstructured/image")

Batch uploaded: 999 images.
Batch uploaded: 99 images.


In [10]:
query = "SELECT * FROM delta_scan('s3://landing-zone/persistent-landing/structured/file_catalog/')"
df_view = con.execute(query).df()
display(df_view)

,file_id,file_path,source_type,file_type,event_time,record_count,metadata_blob,processed_at
0,image_1779622088042.jpg,persistent-landing/unstructured/image/image_1779622088042.jpg,kaggle,Image,2026-05-24 11:32:58.340309,1,"{""label"": ""dew"", ""url"": ""https://www.kaggle.com/datasets/jehanbhathena/weather-dataset"", ""file_size_bytes"": 15451, ""content_type"": ""image/jpg"", ""width"": 240, ""height"": 426, ""aspect_ratio"": 0.56, ""image_mode"": ""RGB"", ""is_corrupted"": false, ""md5"": ""f8ba9864ee0f102f7d5f1970916c311e""}",2026-05-24 11:32:58.340344
1,image_1779622088105.jpg,persistent-landing/unstructured/image/image_1779622088105.jpg,kaggle,Image,2026-05-24 11:32:58.349963,1,"{""label"": ""dew"", ""url"": ""https://www.kaggle.com/datasets/jehanbhathena/weather-dataset"", ""file_size_bytes"": 8081, ""content_type"": ""image/jpg"", ""width"": 205, ""height"": 154, ""aspect_ratio"": 1.33, ""image_mode"": ""RGB"", ""is_corrupted"": false, ""md5"": ""7ddb2866271321a3b2388bb76180a4c1""}",2026-05-24 11:32:58.349996
2,image_1779622088164.jpg,persistent-landing/unstructured/image/image_1779622088164.jpg,kaggle,Image,2026-05-24 11:32:58.359907,1,"{""label"": ""dew"", ""url"": ""https://www.kaggle.com/datasets/jehanbhathena/weather-dataset"", ""file_size_bytes"": 92650, ""content_type"": ""image/jpg"", ""width"": 500, ""height"": 750, ""aspect_ratio"": 0.67, ""image_mode"": ""RGB"", ""is_corrupted"": false, ""md5"": ""bf8c3b3ba818ae289715a9876f6f0867""}",2026-05-24 11:32:58.359939
3,image_1779622088234.jpg,persistent-landing/unstructured/image/image_1779622088234.jpg,kaggle,Image,2026-05-24 11:32:58.368776,1,"{""label"": ""dew"", ""url"": ""https://www.kaggle.com/datasets/jehanbhathena/weather-dataset"", ""file_size_bytes"": 20040, ""content_type"": ""image/jpg"", ""width"": 400, ""height"": 250, ""aspect_ratio"": 1.6, ""image_mode"": ""RGB"", ""is_corrupted"": false, ""md5"": ""8b6f9c58dd8de6ee01173edaf7fb0557""}",2026-05-24 11:32:58.368805
4,image_1779622088296.jpg,persistent-landing/unstructured/image/image_1779622088296.jpg,kaggle,Image,2026-05-24 11:32:58.383996,1,"{""label"": ""dew"", ""url"": ""https://www.kaggle.com/datasets/jehanbhathena/weather-dataset"", ""file_size_bytes"": 245444, ""content_type"": ""image/jpg"", ""width"": 765, ""height"": 746, ""aspect_ratio"": 1.03, ""image_mode"": ""RGB"", ""is_corrupted"": false, ""md5"": ""d46c3c76fd0767528db7555eebc5ed6d""}",2026-05-24 11:32:58.384039
...,...,...,...,...,...,...,...,...
1103,airquality-barcelona.json,persistent-landing/semistructured/airquality-barcelona.json,airquality-barcelona.json,JSON,2026-05-24 11:32:47.994066,15,"{""nesting_level"": 2, ""schema_keys"": [""id"", ""name"", ""locality"", ""timezone"", ""country"", ""owner"", ""provider"", ""isMobile"", ""isMonitor"", ""instruments"", ""sensors"", ""coordinates"", ""licenses"", ""bounds"", ""distance"", ""datetimeFirst"", ""datetimeLast""], ""file_size_bytes"": 27535}",2026-05-24 11:32:47.994086
1104,1779621760.json,persistent-landing/semistructured/airquality-barcelona/2026_05_24/1779621760.json,airquality-barcelona,JSON,2026-05-24 11:22:40.000000,18,"{""nesting_level"": 2, ""schema_keys"": [""id"", ""name"", ""locality"", ""timezone"", ""country"", ""owner"", ""provider"", ""isMobile"", ""isMonitor"", ""instruments"", ""sensors"", ""coordinates"", ""licenses"", ""bounds"", ""distance"", ""datetimeFirst"", ""datetimeLast""], ""file_size_bytes"": 33042}",2026-05-24 11:27:41.701076
1105,1779622038.json,persistent-landing/semistructured/airquality-barcelona/2026_05_24/1779622038.json,airquality-barcelona,JSON,2026-05-24 11:27:18.000000,15,"{""nesting_level"": 2, ""schema_keys"": [""id"", ""name"", ""locality"", ""timezone"", ""country"", ""owner"", ""provider"", ""isMobile"", ""isMonitor"", ""instruments"", ""sensors"", ""coordinates"", ""licenses"", ""bounds"", ""distance"", ""datetimeFirst"", ""datetimeLast""], ""file_size_bytes"": 27535}",2026-05-24 11:27:41.706471
1106,1779621760.json,persistent-landing/